<a href="https://colab.research.google.com/github/dohaalnabahin/Data_science_and_machine_learning_Journey/blob/main/Linear_Regression_in_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##**Import Packages**

In [ ]:
# Import standard packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# Import modeling tools
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer

In [ ]:
# set the default output to pandas
from sklearn import set_config
set_config(transform_output='pandas')

##**Load the Data**

In [ ]:
# Mount google drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# Load the modified galton height data
fpath = "/content/drive/MyDrive/AXSOSACADEMY/02-IntroML/Week06/Data/galton-height-raw.csv"
df = pd.read_csv(fpath)
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 898 entries, 0 to 897
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   family  898 non-null    object 
 1   father  898 non-null    float64
 2   mother  898 non-null    float64
 3   gender  898 non-null    object 
 4   height  898 non-null    float64
 5   kids    898 non-null    int64  
dtypes: float64(3), int64(1), object(2)
memory usage: 42.2+ KB


,family,father,mother,gender,height,kids
0,1,78.5,67.0,M,73.2,4
1,1,78.5,67.0,F,69.2,4
2,1,78.5,67.0,F,69.0,4
3,1,78.5,67.0,F,69.0,4
4,2,75.5,66.5,M,73.5,4


##**Explore the Data**

In [ ]:
# check for duplicate rows
df.duplicated().sum()

np.int64(112)

In [ ]:
# check for null values
df.isna().sum()

,0
family,0
father,0
mother,0
gender,0
height,0
kids,0


In [ ]:
# Checking nuniuqe categories
df.select_dtypes('object').nunique()

,0
family,197
gender,2


In [ ]:
# Drop family colum (high cardinality)
df = df.drop(columns='family')
df.head()

,father,mother,gender,height,kids
0,78.5,67.0,M,73.2,4
1,78.5,67.0,F,69.2,4
2,78.5,67.0,F,69.0,4
3,78.5,67.0,F,69.0,4
4,75.5,66.5,M,73.5,4


In [ ]:
# checking for inconsistent categories
df['gender'].value_counts()

,count
gender,
M,465
F,433


In [ ]:
# checking for inconsistent numeric features
df.describe().round(2)

,father,mother,height,kids
count,898.00,898.00,898.00,898.00
mean,69.23,64.08,66.76,6.14
std,2.47,2.31,3.58,2.69
min,62.00,58.00,56.00,1.00
25%,68.00,63.00,64.00,4.00
50%,69.00,64.00,66.50,6.00
75%,71.00,65.50,69.70,8.00
max,78.50,70.50,79.00,15.00


##**Assign the Target (y) and Features (X)**

In [ ]:
# Separate features vs target & train/test split
X = df.drop(columns = 'height')
y = df['height']

##**Train Test Split**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state = 42)
X_train.head()

,father,mother,gender,kids
377,70.5,62.0,F,8
357,70.5,63.0,F,5
723,67.0,64.0,M,4
306,70.0,64.7,F,7
464,69.0,66.0,F,9


In [ ]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 673 entries, 377 to 102
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   father  673 non-null    float64
 1   mother  673 non-null    float64
 2   gender  673 non-null    object 
 3   kids    673 non-null    int64  
dtypes: float64(2), int64(1), object(1)
memory usage: 26.3+ KB


##**Create the Preprocessor ColumnTransformer**

**StandardScaler:**

بحوّل الأرقام (طول الأب والأم) لأرقام صغيرة متقاربة عشان الموديل ما يتخربط **بالأرقام الكبيرة**

**OneHotEncoder:**

بحوّل كلمة "ذكر/أنثى" لأعمدة (0 و 1) لأن الموديل ما بيفهم نصوص، بيفهم بس رياضيات

In [ ]:
# Get list of numeric columns and instantiate a StandardScaler
num_cols = X_train.select_dtypes('number').columns
scaler = StandardScaler()
# Construct the tuple for column transformer with the scaler
num_tuple = ('numeric',scaler, num_cols)
num_tuple

('numeric',
 StandardScaler(),
 Index(['father', 'mother', 'kids'], dtype='object'))

In [ ]:
# Get list of categorical columns and instantiate a OneHotEncoder
cat_cols = X_train.select_dtypes('object').columns
encoder_ohe = OneHotEncoder(handle_unknown='ignore',sparse_output=False)
# Construct the tuple for column transformer with the encoder
cat_tuple = ('categorical',encoder_ohe, cat_cols)
cat_tuple

('categorical',
 OneHotEncoder(handle_unknown='ignore', sparse_output=False),
 Index(['gender'], dtype='object'))

In [ ]:
# Instantiate the preprocessor/ColumnTransformer
preprocessor = ColumnTransformer([num_tuple, cat_tuple],
                                 verbose_feature_names_out=False)
preprocessor

ColumnTransformer(transformers=[('numeric', StandardScaler(),
                                 Index(['father', 'mother', 'kids'], dtype='object')),
                                ('categorical',
                                 OneHotEncoder(handle_unknown='ignore',
                                               sparse_output=False),
                                 Index(['gender'], dtype='object'))],
                  verbose_feature_names_out=False)

##**Transform the Data with the Preprocessor**

In [ ]:
# Fit the preprocessor on training data
preprocessor.fit(X_train)
# Transform the training and test data
X_train_tf = preprocessor.transform(X_train)
X_test_tf = preprocessor.transform(X_test)
X_train_tf.head()

,father,mother,kids,gender_F,gender_M
377,0.513292,-0.880603,0.706629,1.0,0.0
357,0.513292,-0.458911,-0.412339,1.0,0.0
723,-0.882368,-0.037219,-0.785329,0.0,1.0
306,0.313912,0.257965,0.333640,1.0,0.0
464,-0.084848,0.806165,1.079619,1.0,0.0


#**Step 1: Import and instantiate the model.**

In [ ]:
from sklearn.linear_model import LinearRegression
lin_reg = LinearRegression()
lin_reg

LinearRegression()

#**Step 2: Train the model on your training data.**

# الموديل بيمسك الـ 673 سطر اللي عندك، وبقعد يجرب آلاف الخطوط المستقيمة حتى يوصل لأفضل ميل (Coefficients) وأفضل تقاطع (Intercept) بقلل الغلط بين طول الابن الحقيقي والتوقع.


In [ ]:
# Fit the model on the training data
lin_reg.fit(X_train_tf, y_train)

LinearRegression()

In [ ]:
# View the intercept determined during the fit step
lin_reg.intercept_.round(2)

np.float64(66.69)

In [ ]:
# View the coefficents determined during the fit step
lin_reg.coef_.round(2)

array([ 1.06,  0.69, -0.1 , -2.63,  2.63])

In [ ]:
# Get a list of the feature names after processing
feature_names = preprocessor.get_feature_names_out()
feature_names

array(['father', 'mother', 'kids', 'gender_F', 'gender_M'], dtype=object)

In [ ]:
# Create a Pandas Series with feature name and coefficient
coeffs = pd.Series(lin_reg.coef_.round(2), index = feature_names)
coeffs

,0
father,1.06
mother,0.69
kids,-0.10
gender_F,-2.63
gender_M,2.63


In [ ]:
# View the first row of features in the test set
X_test_tf.head(1)

,father,mother,kids,gender_F,gender_M
331,0.513292,0.173627,-1.904297,1.0,0.0


#**Step 3: Use the model to make predictions for training and testing data**

In [ ]:
# Get predictions for the training data
y_predictions_train = lin_reg.predict(X_train_tf)
# Get predictions for the testing data
y_predictions_test = lin_reg.predict(X_test_tf)

#**Step 4: Evaluate the Results**

**y = 66.69 + (1.06 \ father) + (0.69 \ mother) - (0.10 \ kids) - (2.63 \ gender\_F) + (2.63 \ gender\_M)**

**بتعرفي إنه صح لما يكون الـ Error قريب من الصفر.**

In [ ]:
# Saving a copy of X_test_tf and adding the true and predicted price and the error
prediction_df = X_test_tf.copy()
prediction_df['True Height'] = y_test
prediction_df['Predicted Height'] = y_predictions_test.round(1)
prediction_df['Error'] = (y_predictions_test - y_test).round(1)
prediction_df.head(10)

,father,mother,kids,gender_F,gender_M,True Height,Predicted Height,Error
331,0.513292,0.173627,-1.904297,1.0,0.0,60.0,64.9,4.9
638,-0.483608,-0.458911,0.706629,0.0,1.0,65.5,68.4,2.9
326,0.513292,-0.037219,0.706629,0.0,1.0,68.0,69.8,1.8
848,-1.679888,-0.037219,-0.039350,0.0,1.0,67.0,67.5,0.5
39,1.908952,-0.880603,0.706629,1.0,0.0,63.5,65.4,1.9
327,0.513292,-0.037219,0.706629,0.0,1.0,68.0,69.8,1.8
375,0.513292,-0.880603,0.706629,1.0,0.0,65.0,63.9,-1.1
334,0.313912,-0.037219,-0.785329,1.0,0.0,64.0,64.4,0.4
208,0.712672,-0.880603,-0.412339,0.0,1.0,70.0,69.5,-0.5
136,0.712672,0.806165,-0.785329,1.0,0.0,67.0,65.4,-1.6


In [ ]:
person = pd.DataFrame([[70, 65, 'M', 2]], columns=['father', 'mother', 'gender', 'kids'])

person_tf = preprocessor.transform(person)

prediction = lin_reg.predict(person_tf)
print(f"Predicted Height: {prediction[0].round(1)}")

Predicted Height: 70.1
